# Feature Engineering

In [77]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer
import matplotlib.pyplot as plt

### Loading data

In [78]:
data = pd.read_csv('/Users/christianmonsalve/Desktop/CODE/Projects/Movie_Recs/data/processed/explored_movies.csv')

In [79]:
data.head()

,tconst,titleType,primaryTitle,year,runtimeMinutes,genres,averageRating,numVotes
0,tt0000009,movie,Miss Jerry,1894,45,Romance,5.2,232.0
1,tt0000147,movie,The Corbett-Fitzsimmons Fight,1897,100,"Documentary,News,Sport",5.3,584.0
2,tt0000335,movie,Soldiers of the Cross,1900,40,"Biography,Drama",5.4,67.0
3,tt0000502,movie,Bohemios,1905,100,Unknown,3.1,26.0
4,tt0000574,movie,The Story of the Kelly Gang,1906,70,"Action,Adventure,Biography",6.0,1047.0


## Features

In [80]:
data['hasRating'] = (data['numVotes'] > 0).astype(int)

In [56]:
data['hasRating'].value_counts()

hasRating
0    398510
1    337904
Name: count, dtype: int64

In [57]:
data = data.drop(columns='titleType')

In [58]:
data.head()

,tconst,primaryTitle,year,runtimeMinutes,genres,averageRating,numVotes,hasRating
0,tt0000009,Miss Jerry,1894,45,Romance,5.2,232.0,1
1,tt0000147,The Corbett-Fitzsimmons Fight,1897,100,"Documentary,News,Sport",5.3,584.0,1
2,tt0000335,Soldiers of the Cross,1900,40,"Biography,Drama",5.4,67.0,1
3,tt0000502,Bohemios,1905,100,Unknown,3.1,26.0,1
4,tt0000574,The Story of the Kelly Gang,1906,70,"Action,Adventure,Biography",6.0,1047.0,1


In [59]:
data['genreList'] = data['genres'].str.split(',')

In [60]:
mlb = MultiLabelBinarizer()

In [61]:
genre_encoded = mlb.fit_transform(data['genreList'])

In [62]:
mlb.classes_

array(['Action', 'Adult', 'Adventure', 'Animation', 'Biography', 'Comedy',
       'Crime', 'Documentary', 'Drama', 'Family', 'Fantasy', 'Film-Noir',
       'Game-Show', 'History', 'Horror', 'Music', 'Musical', 'Mystery',
       'News', 'Reality-TV', 'Romance', 'Sci-Fi', 'Short', 'Sport',
       'Talk-Show', 'Thriller', 'Unknown', 'War', 'Western'], dtype=object)

In [63]:
genre_df = pd.DataFrame(data=genre_encoded, columns=mlb.classes_)

In [64]:
data = pd.concat([data, genre_df], axis=1)

In [65]:
data.head()

,tconst,primaryTitle,year,runtimeMinutes,genres,averageRating,numVotes,hasRating,genreList,Action,...,Reality-TV,Romance,Sci-Fi,Short,Sport,Talk-Show,Thriller,Unknown,War,Western
0,tt0000009,Miss Jerry,1894,45,Romance,5.2,232.0,1,[Romance],0,...,0,1,0,0,0,0,0,0,0,0
1,tt0000147,The Corbett-Fitzsimmons Fight,1897,100,"Documentary,News,Sport",5.3,584.0,1,"[Documentary, News, Sport]",0,...,0,0,0,0,1,0,0,0,0,0
2,tt0000335,Soldiers of the Cross,1900,40,"Biography,Drama",5.4,67.0,1,"[Biography, Drama]",0,...,0,0,0,0,0,0,0,0,0,0
3,tt0000502,Bohemios,1905,100,Unknown,3.1,26.0,1,[Unknown],0,...,0,0,0,0,0,0,0,1,0,0
4,tt0000574,The Story of the Kelly Gang,1906,70,"Action,Adventure,Biography",6.0,1047.0,1,"[Action, Adventure, Biography]",1,...,0,0,0,0,0,0,0,0,0,0


In [66]:
data = data.drop(columns=['genres', 'genreList'])

In [67]:
data.loc[:, 'tconst': 'hasRating'].info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 736414 entries, 0 to 736413
Data columns (total 7 columns):
 #   Column          Non-Null Count   Dtype  
---  ------          --------------   -----  
 0   tconst          736414 non-null  object 
 1   primaryTitle    736414 non-null  object 
 2   year            736414 non-null  int64  
 3   runtimeMinutes  736414 non-null  int64  
 4   averageRating   736414 non-null  float64
 5   numVotes        736414 non-null  float64
 6   hasRating       736414 non-null  int64  
dtypes: float64(2), int64(3), object(2)
memory usage: 39.3+ MB


In [68]:
data['logNumVotes'] = np.log1p(data['numVotes'])

In [69]:
data['releaseDecade'] = (data["year"] // 10) * 10

In [70]:
pd.cut(data["runtimeMinutes"], bins=[0, 80, 100, 120, 150, 300]).value_counts()

runtimeMinutes
(0, 80]       436572
(80, 100]     186757
(100, 120]     71124
(120, 150]     31770
(150, 300]     10191
Name: count, dtype: int64

In [71]:
labels = [1, 2, 3, 4, 5]
bins=[0, 80, 100, 120, 150, 300]

data["runtime_bin"] = pd.cut(
    data["runtimeMinutes"],
    bins=bins,
    labels=labels
).astype(int)

In [73]:
m = 50
C = data["averageRating"].mean()

data["bayesRating"] = (
    (data["numVotes"] / (data["numVotes"] + m)) * data["averageRating"]
    + (m / (data["numVotes"] + m)) * C
)

In [74]:
data["bayesRating"].max()

np.float64(9.876764975388367)

In [76]:
data.to_csv('/Users/christianmonsalve/Desktop/CODE/Projects/Movie_Recs/data/processed/new_movie_features.csv', index=False)